# STATE SE — Large Model (PBMC 46k)

Train STATE with the **default (large) architecture** and full context window:
- emsize=512, d_hid=1024, nhead=16, nlayers=8, output_dim=512
- pad_length=2048 (STATE default)
- ESM-2 protein embeddings

Compare: the standard run uses a Geneformer-scale model (emsize=256, d_hid=512, nhead=4, nlayers=3).

## Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.expanduser(
    "~/noise_scaling/modeling/Scaling-up-measurement-noise-scaling-laws/scaling_laws/src"
))

from pathlib import Path
import glob
import numpy as np
import pandas as pd
import anndata as ad
import matplotlib.pyplot as plt
import umap

from scaling_laws.prepare.data import Experiments, PrepareData
from scaling_laws.algo import State

In [ ]:
DATA_DIR = Path(os.path.expanduser("~/noise_scaling/data"))
DATASET = "PBMC"
SIZE = 46_415
QUALITY = 1.0
DEVICE = 4
SEED = 42

## 1. Prepare STATE data

Re-use existing preprocessed state_data (ESM-2 embeddings). Skip if already done.

In [ ]:
state_data = DATA_DIR / DATASET / str(SIZE) / str(QUALITY) / "preprocessed" / "state_data"
profile_name = f"scaling_{DATASET}_{SIZE}_{str(QUALITY).replace('.', '_')}"
marker = state_data / f"all_embeddings_{profile_name}.pt"

if marker.exists():
    print(f"State data already prepared: {marker}")
else:
    experiments = Experiments(
        path_to_data_dir=str(DATA_DIR),
        datasets=[DATASET],
        qualities=[QUALITY],
        sizes=[SIZE],
        algos=["State"],
        signal_columns=["protein_counts"],
        device=DEVICE,
    )
    experiments.prepare_state_data()

## 2. Train (large model)

In [ ]:
base_dir = DATA_DIR / DATASET / str(SIZE) / str(QUALITY)

model = State(
    base_dir=str(base_dir),
    device=DEVICE,
    max_epochs=100,
    early_stopping_patience=3,
    dataset_name=DATASET,
    seed=SEED,
    pad_length=2048,
    # STATE default (large) architecture
    emsize=512,
    d_hid=1024,
    nhead=16,
    nlayers=8,
    output_dim=512,
    batch_size=64,
    max_lr=1e-4,
)

# Save under a distinct results directory
model.save_folder_path = base_dir / "results" / "State_large"
model.model_name = "model"
model.checkpoint_dir = model.save_folder_path / "model" / "checkpoints"
model.embeddings_path = model.save_folder_path / "model" / "embeddings.csv"

print(f"Save folder: {model.save_folder_path}")
print(f"Architecture: emsize={model.emsize}, d_hid={model.d_hid}, "
      f"nhead={model.nhead}, nlayers={model.nlayers}, output_dim={model.output_dim}")
print(f"pad_length={model.pad_length}")

In [ ]:
model.train()

## 3. Training / validation loss curves

In [ ]:
log_dirs = sorted(glob.glob(
    str(model.checkpoint_dir / f"state_{model.profile_name}" / "version_*")
))
metrics_file = Path(log_dirs[-1]) / "metrics.csv"
print(f"Reading: {metrics_file}")

df = pd.read_csv(metrics_file)
train_loss = df[["step", "trainer/train_loss"]].dropna()
val_loss = df[["step", "validation/val_loss"]].dropna()

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(train_loss["step"], train_loss["trainer/train_loss"],
        label="train loss", color="blue", linewidth=1)
ax.plot(val_loss["step"], val_loss["validation/val_loss"],
        color="orange", marker="o", label="val loss", markersize=4, linewidth=1)
ax.set_xlabel("Step")
ax.set_ylabel("Loss")
ax.set_title(f"STATE SE Large — PBMC {SIZE}, q={QUALITY}")
ax.set_xscale("log")
ax.set_yscale("log")
ax.legend()
ax.grid(True, alpha=0.3, which="both", linestyle="--")
fig.tight_layout()
plt.show()

## 4. Embed test set

In [ ]:
embeddings = model.embed()
print(f"Embeddings shape: {embeddings.shape}")

## 5. Compute LMI (protein_counts)

In [ ]:
model.signal_columns = ["protein_counts"]
mi_results = model.mutual_information(max_epochs=300)
print("\nLMI results:")
for signal, mi in mi_results.items():
    print(f"  {signal}: {mi:.5f}")

## 6. Compare LMI across methods

In [ ]:
results_root = base_dir / "results"
algos = [
    ("PCA",              "PCA"),
    ("RandomProjection", "RP"),
    ("SCVI",             "scVI"),
    ("Geneformer",       "Geneformer"),
    ("State",            "State (small)"),
    ("State_large",      "State (large)"),
]

rows = []
for algo_dir, algo_label in algos:
    sig = "Y_protein_counts_1.0_geneformer" if algo_dir == "Geneformer" else "Y_protein_counts_1.0"
    mi_base = results_root / algo_dir / "model" / "MI"
    if not mi_base.exists():
        print(f"{algo_label:20s}  NOT FOUND")
        continue
    for seed_dir in sorted(mi_base.iterdir()):
        mi_file = seed_dir / sig / "lmi_mutual_information.txt"
        if mi_file.exists():
            mi = float(mi_file.read_text().strip())
            rows.append({"Algorithm": algo_label, "seed": int(seed_dir.name), "LMI": mi})
            print(f"{algo_label:20s}  seed={seed_dir.name}  LMI={mi:.5f}")

scores = pd.DataFrame(rows)
scores_agg = (
    scores.groupby("Algorithm")["LMI"]
    .agg(["mean", "std", "count"])
    .rename(columns={"mean": "mean_lmi", "std": "std_lmi", "count": "n_seeds"})
    .reset_index()
    .sort_values("mean_lmi", ascending=False)
)
scores_agg["std_lmi"] = scores_agg["std_lmi"].fillna(0)

colors = {"PCA": "#4C72B0", "RP": "#DD8452", "scVI": "#55A868",
          "Geneformer": "#C44E52", "State (small)": "#8172B3", "State (large)": "#DA8BC3"}

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(
    scores_agg["Algorithm"], scores_agg["mean_lmi"],
    yerr=scores_agg["std_lmi"], capsize=4,
    color=[colors.get(a, "#999") for a in scores_agg["Algorithm"]],
    edgecolor="black", linewidth=0.5,
)
for bar, mean, std, n in zip(bars, scores_agg["mean_lmi"], scores_agg["std_lmi"], scores_agg["n_seeds"]):
    label = f"{mean:.3f}"
    if n > 1:
        label += f"\n\u00b1{std:.3f} (n={n})"
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + std + 0.02,
            label, ha="center", va="bottom", fontsize=9)
ax.set_ylabel("LMI (protein_counts)")
ax.set_title(f"PBMC {SIZE}, q={QUALITY} — LMI comparison (State large vs others)")
ax.grid(axis="y", alpha=0.3)
ax.set_ylim(0, (scores_agg["mean_lmi"] + scores_agg["std_lmi"]).max() * 1.2)
plt.xticks(rotation=15, ha="right")
fig.tight_layout()
plt.show()

## 7. UMAP

In [ ]:
max_cells = 10_000
if embeddings.shape[0] > max_cells:
    rng = np.random.default_rng(42)
    idx = rng.choice(embeddings.shape[0], max_cells, replace=False)
    emb_sub = embeddings[idx]
else:
    emb_sub = embeddings

reducer = umap.UMAP(n_components=2, random_state=42, n_jobs=1)
umap_coords = reducer.fit_transform(emb_sub)

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(umap_coords[:, 0], umap_coords[:, 1], color="#888", s=2, alpha=0.7)
ax.set_title(f"STATE SE (large) — PBMC {SIZE}, q={QUALITY}")
ax.set_xlabel("UMAP 1")
ax.set_ylabel("UMAP 2")
fig.tight_layout()
plt.show()